In [37]:
import torch
import numpy as np

from utils import *
from src.iipw import *


d1 = 10000
d2 = 1000
r = 10
p = 0.05
ob = 50
sample = "iid"
if torch.cuda.is_available():
    device = 'cuda:1'
else:
    device = 'cpu'

# dataset
dataset = "syn"
#M = load_normalized_data_syn(r, d1, d2, device)
#M = load_data_syn(r, d1, d2, device)
#M = rowwise_normalize(M)
err_T_list = []
err_T_p_list = []
M_mean_list = []
M_var_list = []
M = load_normalized_data_syn(r, d1, d2, device)
MTM = M.T @ M
runs = 50

A_list = []
T_list = []
T_err_list = []
for run in tqdm(range(runs)):

    if sample == "iid":
        # IID sample
        observed_M, masks = get_uniform_masks(M, p)
    else:
        # few entry sample
        observed_M, masks = get_random_samples_per_row(M.cpu().numpy(), ob)
        p = ob/d2
        observed_M = torch.from_numpy(observed_M).float().to(device)
        masks = torch.from_numpy(masks).to(device)
    
    
    normalized_MTM = MTM / d1
    A =  observed_M.T @ observed_M

    B = (1 * (observed_M != 0)).float().T @ (1 * (observed_M != 0).float())
    B = B + (B == 0) * 1

    A_list.append(A.unsqueeze(0))
    T = iipw_T(observed_M)
    T_list.append(T.unsqueeze(0).cpu())
    T_err_list.append(compute_err_tensor(estimation_matrix=T, groundtruth_matrix=normalized_MTM).item())



var_T = compute_var(T_list)[0]

diag_mask = ~torch.eye(var_T.size(0), dtype=bool)

var_T_diag = torch.diag(var_T)
var_T_diag_mean = var_T_diag.mean()
var_T_off_diag = var_T[diag_mask]
var_T_off_diag_mean = var_T_off_diag.mean()




MTM_diag = torch.diag(MTM)
MTM_off_diag = MTM[diag_mask]
MTM_diag_mean = MTM_diag.mean()
MTM_off_diag_mean = MTM_off_diag.mean()

print(var_T.shape)
#var_T_value = (var_T).mean()
var_T_value = var_T[2,3]

var_A = compute_var(A_list)[0]
var_A_diag = torch.diag(var_A)
var_A_off_diag = var_A[diag_mask]
var_A_diag_mean = var_A_diag.mean()
var_A_off_diag_mean = var_A_off_diag.mean()

#print(MTM_off_diag)
#print(MTM_diag)
estimate_offdiag = var_A_off_diag / (d1**2 * p**4) - ((1-p**2) * (MTM_off_diag**2)) / (d1**3 * p**2)
estimate_diag = var_A_diag / (d1**2 * p**2) - ((1-p) * (MTM_diag**2)) / (d1**3 * p)


print("estimate diag", estimate_diag.mean())
print("estimate off diag", estimate_offdiag.mean())
print("var T diag", var_T_diag_mean)
print("var T off diag", var_T_off_diag_mean)





100%|██████████| 50/50 [00:04<00:00, 11.12it/s]


=========> Computing variance
average count: 50.0
minimum count: 50.0
torch.Size([1000, 1000])
=========> Computing variance
average count: 50.0
minimum count: 50.0
estimate diag tensor(-7.2972e-10, device='cuda:1')
estimate off diag tensor(-1.2897e-08, device='cuda:1')
var T diag tensor(4.8910e-13)
var T off diag tensor(5.6486e-12)
